In [2]:
# Connect to the Cleveland_Clinic MySQL database.
# quote_plus() safely encodes special characters (like '@') in the password
# so SQLAlchemy's connection URL doesn't misparse it as a host separator.
from sqlalchemy import create_engine
from urllib.parse import quote_plus

DB_USER = "root"
DB_PASSWORD = quote_plus("YOUR_PASSWORD")   # encodes special characters safely
DB_HOST = "localhost"
DB_NAME = "Cleveland_Clinic"

engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

import pandas as pd
test = pd.read_sql("SHOW TABLES;", engine)
print(test)

  Tables_in_cleveland_clinic
0               appointments
1                    billing
2                   branches
3       doctor_branch_visits
4                    doctors
5                lab_reports
6                   patients
7              prescriptions
8                  surgeries


In [3]:
# Load branches.csv first - it has no foreign key dependencies on any
# other table, so it's safe to load before anything else.
df_branches = pd.read_csv('branches.csv').rename(columns={
    'Branch ID': 'branch_id', 'Location': 'location',
    'OT_Available': 'ot_available', 'Inhouse Anesthetician': 'inhouse_anesthetician',
    'No. of OTs': 'no_of_ots', 'Inhouse Pharmacy': 'inhouse_pharmacy',
    'Inhouse Nutritionist': 'inhouse_nutritionist', 'Inpatient Rooms': 'inpatient_rooms',
    'Luxury Suite Rooms': 'luxury_suite_rooms', 'Prayer Room': 'prayer_room',
    'Kids Play Area': 'kids_play_area',
})

cols = ['branch_id', 'location', 'ot_available', 'inhouse_anesthetician',
        'no_of_ots', 'inhouse_pharmacy', 'inhouse_nutritionist',
        'inpatient_rooms', 'luxury_suite_rooms', 'prayer_room', 'kids_play_area']

df_branches[cols].to_sql('branches', engine, if_exists='append', index=False)
print("Done")

Done


In [4]:
# Load doctors.csv. Keep the RAW (unrenamed) version in df_doctors_raw
# too, since the next cell needs its original 'Other Branches Visited'
# column to build the junction table.
df_doctors_raw = pd.read_csv('doctors.csv')

df_doctors = df_doctors_raw.rename(columns={
    'Doctor Name': 'doctor_name', 'Doctor ID': 'doctor_id',
    'Specialization': 'specialization', 'Qualification': 'qualification',
    'Surgery Experience': 'surgery_experience', 'Practicing Branch': 'practicing_branch',
})

cols = ['doctor_id', 'doctor_name', 'specialization', 'qualification',
        'surgery_experience', 'practicing_branch']

df_doctors[cols].to_sql('doctors', engine, if_exists='append', index=False)
print("Done")

Done


In [5]:
# Normalize doctors.csv's 'Other Branches Visited' column, which stores
# a Python-list-as-string per row (e.g. "['FR33','MA60']") - a 1NF
# violation. ast.literal_eval() parses it into a real list, then
# .explode() turns each doctor's list into one row per doctor-branch pair.
import ast

df_visits = df_doctors_raw[['Doctor ID', 'Other Branches Visited']].copy()
df_visits['Other Branches Visited'] = df_visits['Other Branches Visited'].apply(ast.literal_eval)

df_visits_exploded = df_visits.explode('Other Branches Visited').rename(columns={
    'Doctor ID': 'doctor_id', 'Other Branches Visited': 'branch_id',
})

df_visits_exploded.to_sql('doctor_branch_visits', engine, if_exists='append', index=False)
print("Done")

Done


In [6]:
# Load patients.csv. All three date columns need explicit format
# conversion. future_visit_flag marks the 134 rows where
# 'Date of Latest Consultation' is impossibly in the future - flagged
# rather than deleted, so raw data is preserved for audit.
df_patients = pd.read_csv('patients.csv').rename(columns={
    'Patient ID': 'patient_id', 'Date of Registration': 'date_of_registration',
    'Date of First Consultation': 'date_of_first_consultation',
    'Date of Latest Consultation': 'date_of_latest_consultation',
    'Overall Satisfaction Rating': 'overall_satisfaction_rating',
    'Admission': 'admission', 'Surgery Underwent': 'surgery_underwent',
    'Total Number of Visits': 'total_number_of_visits',
})

for col in ['date_of_registration', 'date_of_first_consultation', 'date_of_latest_consultation']:
    df_patients[col] = pd.to_datetime(df_patients[col], format='%m/%d/%Y')

df_patients['future_visit_flag'] = df_patients['date_of_latest_consultation'] > pd.Timestamp.today()

cols = ['patient_id', 'date_of_registration', 'date_of_first_consultation',
        'date_of_latest_consultation', 'overall_satisfaction_rating', 'admission',
        'surgery_underwent', 'total_number_of_visits', 'future_visit_flag']

df_patients[cols].to_sql('patients', engine, if_exists='append', index=False)
print("Done")

Done


In [7]:
# Load surgeries.csv. This table is the 'parent' for billing and
# lab_reports (both reference its case_id), so it must load before them.
df_surgeries = pd.read_csv('surgeries.csv').rename(columns={
    'Case ID': 'case_id', 'Serial No': 'serial_no', 'Patient ID': 'patient_id',
    'Doctor ID': 'doctor_id', 'Branch ID': 'branch_id', 'Surgery Type': 'surgery_type',
    'Surgery Date': 'surgery_date', 'Anesthesia Type': 'anesthesia_type',
    'Surgery Outcome': 'surgery_outcome', 'Surgery Duration (hours)': 'surgery_duration_hours',
    'Recovery Period (days)': 'recovery_period_days',
})

df_surgeries['surgery_date'] = pd.to_datetime(df_surgeries['surgery_date'], format='%m/%d/%Y')

cols = ['case_id', 'serial_no', 'patient_id', 'doctor_id', 'branch_id', 'surgery_type',
        'surgery_date', 'anesthesia_type', 'surgery_outcome', 'surgery_duration_hours',
        'recovery_period_days']

df_surgeries[cols].to_sql('surgeries', engine, if_exists='append', index=False)
print("Done")

Done


In [8]:
# Load billing.csv. case_id here is a real, validated foreign key
# back to surgeries.case_id, so surgeries must already be loaded.
df_billing = pd.read_csv('billing.csv').rename(columns={
    'Claim ID': 'claim_id', 'Serial No': 'serial_no', 'Patient ID': 'patient_id',
    'Case ID': 'case_id', 'Branch ID': 'branch_id', 'Total Bill': 'total_bill',
    'Insurance Provider': 'insurance_provider', 'Amount Covered': 'amount_covered',
    'Out-of-Pocket Expenses': 'out_of_pocket_expenses', 'Claim Status': 'claim_status',
})

cols = ['claim_id', 'serial_no', 'patient_id', 'case_id', 'branch_id', 'total_bill',
        'insurance_provider', 'amount_covered', 'out_of_pocket_expenses', 'claim_status']

df_billing[cols].to_sql('billing', engine, if_exists='append', index=False)
print("Done")

Done


In [9]:
# Load lab_reports.csv. Same as billing - case_id is a real foreign
# key back to surgeries.case_id.
df_lab_reports = pd.read_csv('lab_reports.csv').rename(columns={
    'Report ID': 'report_id', 'Serial No': 'serial_no', 'Case ID': 'case_id',
    'Patient ID': 'patient_id', 'Branch ID': 'branch_id', 'Test Name': 'test_name',
    'Test Date': 'test_date', 'Result': 'result',
})

df_lab_reports['test_date'] = pd.to_datetime(df_lab_reports['test_date'], format='%m/%d/%Y')

cols = ['report_id', 'serial_no', 'case_id', 'patient_id', 'branch_id',
        'test_name', 'test_date', 'result']

df_lab_reports[cols].to_sql('lab_reports', engine, if_exists='append', index=False)
print("Done")

Done


In [10]:
# Load appointments.csv. No foreign keys enforced here - validation
# showed its Case ID and Doctor ID columns don't reliably link to
# other tables (confirmed by direct set-overlap checks).
df_appointments = pd.read_csv('appointments.csv').rename(columns={
    'Serial No': 'serial_no', 'Case ID': 'case_id', 'Patient ID': 'patient_id',
    'Branch ID': 'branch_id', 'Date of Consultation': 'date_of_consultation',
    'Requested Doctor ID': 'requested_doctor_id', 'Consulted Doctor ID': 'consulted_doctor_id',
    'Reason for Visit': 'reason_for_visit', 'Diagnosis': 'diagnosis',
    'Prescribed Medication': 'prescribed_medication', 'Followup Required': 'followup_required',
    'Nutritionist Recommended': 'nutritionist_recommended',
})

df_appointments['date_of_consultation'] = pd.to_datetime(df_appointments['date_of_consultation'], format='%m/%d/%Y')

cols = ['serial_no', 'case_id', 'patient_id', 'branch_id', 'date_of_consultation',
        'requested_doctor_id', 'consulted_doctor_id', 'reason_for_visit', 'diagnosis',
        'prescribed_medication', 'followup_required', 'nutritionist_recommended']

df_appointments[cols].to_sql('appointments', engine, if_exists='append', index=False)
print("Done")

Done


In [11]:
# Load prescriptions.csv. No foreign key on doctor_id - validation
# showed 100% of its Doctor IDs don't exist in doctors.csv, despite
# matching character length by coincidence.
df_prescriptions = pd.read_csv('prescriptions.csv').rename(columns={
    'Bill ID': 'bill_id', 'Serial No': 'serial_no', 'Patient ID': 'patient_id',
    'Doctor ID': 'doctor_id', 'Medication Name': 'medication_name', 'Dosage': 'dosage',
    'Quantity': 'quantity', 'Pharmacy Availability': 'pharmacy_availability',
    'Issued Date': 'issued_date',
})

df_prescriptions['issued_date'] = pd.to_datetime(df_prescriptions['issued_date'], format='%m/%d/%Y')

cols = ['bill_id', 'serial_no', 'patient_id', 'doctor_id', 'medication_name',
        'dosage', 'quantity', 'pharmacy_availability', 'issued_date']

df_prescriptions[cols].to_sql('prescriptions', engine, if_exists='append', index=False)
print("Done")

Done


## Verification: confirm all tables loaded correctly

Expected: branches=100, doctors=200, doctor_branch_visits=384, patients=2300 (134 flagged), surgeries=1500, billing=5000, lab_reports=7500, appointments=12000, prescriptions=9000


In [ ]:
# Final verification: confirm every table loaded the expected row
# count, and that the future_visit_flag was applied to the correct
# number of patient rows (134).
tables = ['branches', 'doctors', 'doctor_branch_visits', 'patients',
          'surgeries', 'billing', 'lab_reports', 'appointments', 'prescriptions']

for t in tables:
    count = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {t}", engine)['cnt'][0]
    print(f"{t:25s} {count}")

flagged = pd.read_sql("SELECT COUNT(*) AS cnt FROM patients WHERE future_visit_flag = 1", engine)['cnt'][0]
print(f"\nPatients flagged with future visit date: {flagged}")
